# 02 — Limpieza e integración
StreamView Analytics · ADY1104 · Matías Retamal · Claudio González

Este notebook **no contiene lógica de transformación**: importa las funciones de `src/limpieza.py` y las ejecuta paso a paso, mostrando el efecto de cada una. La lógica vive en el módulo para que el pipeline sea reproducible y testeable; el notebook es la narración de lo que el módulo hace.

Entradas: `data/raw/` (nunca se modifica) · Salidas: `data/processed/`

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
from limpieza import (
    cargar_datos, prefijar_ids, eliminar_columnas_inservibles, agregar_tipo,
    unificar, explotar, agregar_metricas, construir_catalogo, exportar,
    COLUMNAS_ELIMINADAS, UMBRAL_VOTOS,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

peliculas, series = cargar_datos()
print('películas:', peliculas.shape)
print('series:   ', series.shape)

## Paso 1 — Prefijo de IDs (`MOV_` / `TV_`)

Los 397 `show_id` que colisionan entre archivos apuntan a títulos distintos. Sin prefijo, la concatenación los fusiona.

In [ ]:
colisiones = set(peliculas['show_id']) & set(series['show_id'])
ejemplo = list(colisiones)[0]
print('show_id en conflicto:', ejemplo)
print('  en películas →', peliculas.loc[peliculas.show_id == ejemplo, 'title'].iloc[0])
print('  en series    →', series.loc[series.show_id == ejemplo, 'title'].iloc[0])

p = prefijar_ids(peliculas, 'MOV_')
s = prefijar_ids(series, 'TV_')
print('\ntras prefijar, colisiones restantes:', len(set(p['show_id']) & set(s['show_id'])))

## Paso 2 — Eliminar `rating` y `duration`

Las dos columnas se eliminan porque no aportan información, no por comodidad. El motivo de cada una está documentado en el propio módulo.

In [ ]:
for columna, motivo in COLUMNAS_ELIMINADAS.items():
    print(f'{columna:10s} → {motivo}')

p = eliminar_columnas_inservibles(p)
s = eliminar_columnas_inservibles(s)
print('\ncolumnas tras la limpieza:')
print(' películas:', list(p.columns))
print(' series:   ', list(s.columns))

## Paso 3 — Columna `tipo` y concatenación

`type` ya existía con valores `Movie` / `TV Show`: no se crea información nueva, se normaliza al idioma de la audiencia. Los 9 duplicados internos de series sobreviven al prefijo (mismo id dentro del mismo archivo) y se resuelven al concatenar.

In [ ]:
catalogo = unificar(peliculas, series)

print('filas esperadas sin deduplicar:', len(peliculas) + len(series))
print('filas del catálogo unificado: ', len(catalogo))
print('diferencia (duplicados internos de series):',
      len(peliculas) + len(series) - len(catalogo))
print()
print(catalogo['tipo'].value_counts().to_string())

## Paso 4 — Métricas derivadas

**`roi = revenue / budget`**, solo donde ambos son positivos. Es un multiplicador: 1.0 es punto de equilibrio. Queda nulo en las series (no traen datos financieros) y en el 78% de las películas.

**`score_ponderado`**, fórmula tipo IMDb, para que un 10.0 con un voto no encabece ningún ranking.

In [ ]:
catalogo = agregar_metricas(catalogo)

print('umbral de votos por tipo:', UMBRAL_VOTOS)
print('títulos con votación suficiente:')
print(catalogo.groupby('tipo')['votos_suficientes'].agg(['sum', 'size']).to_string())
print('\npelículas con ROI calculable:', int(catalogo['roi'].notna().sum()))

### Por qué el `score_ponderado` y no la nota cruda

La comparación de abajo es la justificación del indicador en una sola celda: el ranking por `vote_average` está tomado por títulos con uno o dos votos.

In [ ]:
cols = ['title', 'tipo', 'vote_count', 'vote_average', 'score_ponderado']

print('TOP 5 por nota cruda (vote_average)')
print(catalogo.nlargest(5, 'vote_average')[cols].to_string(index=False))

print('\nTOP 5 por score ponderado')
print(catalogo.nlargest(5, 'score_ponderado')[cols].to_string(index=False))

## Paso 5 — Tablas largas: género, país y reparto

El 26% de las películas lista más de un país y casi todos los títulos listan varios géneros. Agregar sobre la columna cruda contaría `"United States of America, France"` como una categoría propia.

Las tablas largas se guardan **aparte**: la tabla principal conserva una fila por título.

In [ ]:
genero_largo = explotar(catalogo, 'genres', 'genero')

print('una fila por título-género:')
print(genero_largo.head(6).to_string(index=False))
print('\nfilas:', len(genero_largo), '| géneros distintos:', genero_largo['genero'].nunique())

## Paso 6 — Pipeline completo y exportación

`construir_catalogo()` encadena todo lo anterior en una llamada. El notebook se puede correr de arriba a abajo sin intervención manual.

In [ ]:
catalogo, largos = construir_catalogo()
rutas = exportar(catalogo, largos)

print(f'catálogo unificado: {catalogo.shape[0]:,} filas × {catalogo.shape[1]} columnas\n')
for nombre, tabla in largos.items():
    print(f'  {nombre:8s} {len(tabla):>7,} filas | {tabla.iloc[:, -1].nunique():>5,} valores distintos')
print()
for nombre, ruta in rutas.items():
    print(f'  → {ruta.name}')

In [ ]:
catalogo.head()

---
## Resultado

`data/processed/catalogo_unificado.csv` — 31.991 títulos, una fila por título, listo para el análisis exploratorio (notebook 03) y para el dashboard.

Los tres archivos largos (`catalogo_genero.csv`, `catalogo_pais.csv`, `catalogo_actor.csv`) se usan para agregar por esas dimensiones sin inflar el conteo de títulos.